In [6]:
import pandas as pd
from datasets import load_dataset
import re
import nltk

### Extraccción de dataset y filtrado

In [7]:
# FASE 1: EXTRACCIÓN Y BÚSQUEDA BIDIRECCIONAL
print("1. Descargando el corpus masivo EMEA...")
dataset = load_dataset("emea", lang1="en", lang2="es", split="train", trust_remote_code=True)


df = pd.DataFrame({
    'Texto_Ingles': [item['en'] for item in dataset['translation']],
    'Texto_Espanol': [item['es'] for item in dataset['translation']]
})

print("\n2. Aplicando Extracción Médica Especializada...")

keywords_en = r"(breast cancer|breast tumor|tumor|neoplasm|mammary|carcinoma|mastectom|oncolog|brca[12]?|her2|tamoxifen|trastuzumab|metastas|lymph|estrogen|chemotherap|malignant|cancer)"


keywords_es = r"(c[aá]ncer de mama|tumor de mama|tumor|neoplasia|mamari[ao]|carcinoma|mastectom[ií]a|oncolog[ií]a|brca[12]?|her2|tamoxifeno|trastuzumab|met[aá]stas|ganglio|linf[aá]tico|estr[oó]geno|quimioterap|malign[oa]|c[aá]ncer)"

# Pre-compilación del Regex
pattern_en = re.compile(keywords_en, re.IGNORECASE)
pattern_es = re.compile(keywords_es, re.IGNORECASE)

# Máscara de búsqueda en cualquiera de los dos idiomas
mask_en = df['Texto_Ingles'].str.contains(pattern_en, na=False)
mask_es = df['Texto_Espanol'].str.contains(pattern_es, na=False)

# Unimos los filtros, quitamos nulos y eliminamos duplicados idénticos
df_unico = df[mask_en | mask_es].dropna().drop_duplicates()
print(f"-> Filas atrapadas inicialmente: {len(df_unico):,}")


# FASE 2: LIMPIEZA DE RUIDO Y BASURA WEB

print("\n3. Purgando basura de HTML y URLs...")
mascara_basura_en = ~df_unico['Texto_Ingles'].str.contains(r'<[^>]+>|http', regex=True, na=False)
mascara_basura_es = ~df_unico['Texto_Espanol'].str.contains(r'<[^>]+>|http', regex=True, na=False)

df_limpio = df_unico[mascara_basura_en & mascara_basura_es].copy()
print(f"-> Filas que sobrevivieron a la purga de ruido: {len(df_limpio):,}")




nombre_archivo = "Corpus_final.csv"
df_limpio.to_csv(nombre_archivo, index=False, encoding='utf-8')





# Muestra aleatoria visual (si estás en Jupyter, puedes usar display() en lugar de print)
print("\n--- MUESTRA ALEATORIA (5 FILAS) ---")
display(df_limpio.sample(5))

1. Descargando el corpus masivo EMEA...

2. Aplicando Extracción Médica Especializada...


C:\Users\PC\AppData\Local\Temp\ipykernel_28928\1410543704.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_en = df['Texto_Ingles'].str.contains(pattern_en, na=False)
C:\Users\PC\AppData\Local\Temp\ipykernel_28928\1410543704.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_es = df['Texto_Espanol'].str.contains(pattern_es, na=False)


-> Filas atrapadas inicialmente: 7,340

3. Purgando basura de HTML y URLs...
-> Filas que sobrevivieron a la purga de ruido: 7,316

--- MUESTRA ALEATORIA (5 FILAS) ---


,Texto_Ingles,Texto_Espanol
1009503,Both groups may be at increased risk of Neurol...,Ambos grupos pueden aumentar el riesgo de Sínd...
507422,These receptors are found in large numbers on ...,Estos receptores se encuentran en grandes cant...
571030,Studies in vitro and in animal models demonstr...,Estudios in vitro y en modelos en animales dem...
211970,The safety of EVISTA in patients with breast c...,No se ha estudiado de forma adecuada la seguri...
507440,The Committee for Medicinal products for Human...,El Comité de Medicamentos de Uso Humano (CHMP)...


### Limpieza de duplicados y valores nulos.

In [8]:
print("1. Cargando el dataset crudo...")

df = pd.read_csv("Corpus_final.csv")

# Guardamos el número de filas originales para comparar al final
total_original = len(df)
print(f"Filas originales: {total_original:,}")

print("\n2. Limpiando valores nulos y duplicados (Doble Check)...")
# Eliminar filas donde falte el texto en inglés o en español
df_limpio = df.dropna()

# Eliminar filas que sean exactamente iguales (duplicados)
df_limpio = df_limpio.drop_duplicates()

total_limpio = len(df_limpio)
print(f"Filas después de limpiar: {total_limpio:,}")
print(f"¡Se eliminaron {total_original - total_limpio:,} filas inservibles o repetidas!")

print("\n3. Realizando Análisis Descriptivo (Conteo de Tokens)")
# CORRECCIÓN: Usamos los nombres exactos de las columnas generadas en la Fase 1
df_limpio['Tokens_Ingles'] = df_limpio['Texto_Ingles'].apply(lambda x: len(str(x).split()))
df_limpio['Tokens_Espanol'] = df_limpio['Texto_Espanol'].apply(lambda x: len(str(x).split()))

# Extraemos las estadísticas para Inglés
max_en = df_limpio['Tokens_Ingles'].max()
min_en = df_limpio['Tokens_Ingles'].min()
promedio_en = round(df_limpio['Tokens_Ingles'].mean(), 2)

# Extraemos las estadísticas para Español
max_es = df_limpio['Tokens_Espanol'].max()
min_es = df_limpio['Tokens_Espanol'].min()
promedio_es = round(df_limpio['Tokens_Espanol'].mean(), 2)

print("\n---ANÁLISIS DESCRIPTIVO ---")
print(f"INGLÉS  -> Max tokens: {max_en} | Min tokens: {min_en} | Promedio: {promedio_en} tokens/oración")
print(f"ESPAÑOL -> Max tokens: {max_es} | Min tokens: {min_es} | Promedio: {promedio_es} tokens/oración")

print("\n4. Guardando el dataset final limpio...")
# Guardamos este nuevo dataset perfecto en un CSV nuevo
df_limpio.to_csv("Banco_de_Datos_Limpio.csv", index=False, encoding='utf-8')
print("archivo 'Banco_de_Datos_Limpio.csv' ha sido guardado.")

1. Cargando el dataset crudo...
Filas originales: 7,316

2. Limpiando valores nulos y duplicados (Doble Check)...
Filas después de limpiar: 7,316
¡Se eliminaron 0 filas inservibles o repetidas!

3. Realizando Análisis Descriptivo (Conteo de Tokens)

---ANÁLISIS DESCRIPTIVO ---
INGLÉS  -> Max tokens: 386 | Min tokens: 1 | Promedio: 26.62 tokens/oración
ESPAÑOL -> Max tokens: 444 | Min tokens: 1 | Promedio: 30.86 tokens/oración

4. Guardando el dataset final limpio...
archivo 'Banco_de_Datos_Limpio.csv' ha sido guardado.


### Filtrado Estadístico y Remoción de Valores Atípicos


In [9]:
# Volvemos a contar los tokens rápido
# CORRECCIÓN: Usamos el nombre correcto 'Texto_Ingles'
df_limpio['Tokens_Ingles'] = df_limpio['Texto_Ingles'].apply(lambda x: len(str(x).split()))

#Calcular los percentiles

max_ideal = int(df_limpio['Tokens_Ingles'].quantile(0.98))

# minimo ideal quitamos el 2% más bajo para quitar oraciones basura como de 1 o 2 palabras
min_ideal = int(df_limpio['Tokens_Ingles'].quantile(0.02))

print(f"--- CÁLCULO ESTADÍSTICO DE LÍMITES ---")
print(f"Para mantener el 96% de los datos más útiles del dataset:")
print(f"MIN ideal : {min_ideal} tokens")
print(f"MAX ideal : {max_ideal} tokens")



print("\nFiltrando el dataset con los nuevos límites...")
# Nos quedamos solo con las filas que estén entre tu min_ideal y max_ideal
df_final = df_limpio[
    (df_limpio['Tokens_Ingles'] >= min_ideal) &
    (df_limpio['Tokens_Ingles'] <= max_ideal)
]

print(f"Oraciones listas para el modelo IBM 1: {len(df_final):,}")
df_final.to_csv("Corpus_IBM_Listo.csv", index=False, encoding='utf-8')
print("¡El CSV definitivo ha sido creado exitosamente!")

--- CÁLCULO ESTADÍSTICO DE LÍMITES ---
Para mantener el 96% de los datos más útiles del dataset:
MIN ideal : 2 tokens
MAX ideal : 92 tokens

Filtrando el dataset con los nuevos límites...
Oraciones listas para el modelo IBM 1: 7,098
¡El CSV definitivo ha sido creado exitosamente!


### Normalizacion y Tokenización(NLTK)

In [11]:
nltk.download('punkt')
nltk.download('punkt_tab')

print("1. Cargando el corpus filtrado (Materia prima)...")
df = pd.read_csv("Corpus_IBM_Listo.csv")

print("\n2. Aplicando Normalización (Híbrida: ESCOM + Dominio Médico)...")
df['Texto_Ingles'] = df['Texto_Ingles'].str.lower()
df['Texto_Espanol'] = df['Texto_Espanol'].str.lower()

# Quitamos puntuación pero cambiándola por un espacio
df['Texto_Ingles'] = df['Texto_Ingles'].apply(lambda x: re.sub(r'[^\w\s]', ' ', str(x)))
df['Texto_Espanol'] = df['Texto_Espanol'].apply(lambda x: re.sub(r'[^\w\s]', ' ', str(x)))

# Limpiamos los espacios múltiples que dejaron las comas y puntos
df['Texto_Ingles'] = df['Texto_Ingles'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())
df['Texto_Espanol'] = df['Texto_Espanol'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())

print("\n3.Tokenización (NLTK)...")
df['Tokens_Ingles'] = df['Texto_Ingles'].apply(nltk.word_tokenize)
df['Tokens_Espanol'] = df['Texto_Espanol'].apply(nltk.word_tokenize)

print("\n--- MUESTRA DEL PREPROCESAMIENTO ---")
display(df[['Texto_Ingles', 'Tokens_Ingles']].sample(3))

df.to_csv("Corpus_Tokenizado_Final.csv", index=False, encoding='utf-8')
print("\n¡Listo! El archivo 'Corpus_Tokenizado_Final.csv' está preparado para el Modelo IBM 1.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


1. Cargando el corpus filtrado (Materia prima)...

2. Aplicando Normalización (Híbrida: ESCOM + Dominio Médico)...

3.Tokenización (NLTK)...

--- MUESTRA DEL PREPROCESAMIENTO ---


,Texto_Ingles,Tokens_Ingles
5251,the committee for medicinal products for human...,"[the, committee, for, medicinal, products, for..."
4637,during the first 21 day chemotherapy cycle pat...,"[during, the, first, 21, day, chemotherapy, cy..."
6378,research non governmental organisations in fra...,"[research, non, governmental, organisations, i..."



¡Listo! El archivo 'Corpus_Tokenizado_Final.csv' está preparado para el Modelo IBM 1.
